# Tako HRM - Setup

**Run this notebook once per Colab session to set up the environment.**

This notebook:
- Installs uv package manager
- Clones the tako-v2 repository (public)
- Mounts Google Drive for checkpoint persistence
- Checks GPU availability
- Installs dependencies

After running this, you can use `train.ipynb`, `eval.ipynb`, `play.ipynb`, and `benchmark.ipynb`.

---

## Step 1: Install uv Package Manager

In [1]:
# Install uv (Rust-based Python package manager)
!curl -LsSf https://astral.sh/uv/install.sh | sh

# Verify installation (uv installs to /usr/local/bin in Colab)
!uv --version

print("\n✅ uv installed successfully")

downloading uv 0.10.9 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv 0.10.9

✅ uv installed successfully


## Step 2: Clone Repository

In [2]:
# Clone public tako-v2 repository
import os

REPO_URL = "https://github.com/dosequus/tako-v2.git"
REPO_NAME = "tako-v2"
os.chdir('/content')
if os.path.exists(REPO_NAME):
    print(f"✅ Repository already exists at: {os.path.abspath(REPO_NAME)}")
    print("   Updating to latest version...")
    !cd {REPO_NAME} && git pull
else:
    print(f"📥 Cloning repository from: {REPO_URL}")
    !git clone {REPO_URL}
    print(f"\n✅ Repository cloned to: {os.path.abspath(REPO_NAME)}")

# Change to repo directory
os.chdir(REPO_NAME)
print(f"\n📂 Working directory: {os.getcwd()}")

📥 Cloning repository from: https://github.com/dosequus/tako-v2.git
Cloning into 'tako-v2'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 172 (delta 88), reused 132 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 1003.74 KiB | 5.67 MiB/s, done.
Resolving deltas: 100% (88/88), done.

✅ Repository cloned to: /content/tako-v2

📂 Working directory: /content/tako-v2


## Step 3: Mount Google Drive

This persists checkpoints across sessions.

In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory structure
!mkdir -p /content/drive/MyDrive/tako_checkpoints/tictactoe
!mkdir -p /content/drive/MyDrive/tako_checkpoints/othello
!mkdir -p /content/drive/MyDrive/tako_checkpoints/hex
!mkdir -p /content/drive/MyDrive/tako_checkpoints/chess

# Link to repo
!rm -rf checkpoints
!ln -s /content/drive/MyDrive/tako_checkpoints checkpoints

print("\n✅ Google Drive mounted")
print("✅ Checkpoints will be saved to: /content/drive/MyDrive/tako_checkpoints")

Mounted at /content/drive

✅ Google Drive mounted
✅ Checkpoints will be saved to: /content/drive/MyDrive/tako_checkpoints


## Step 4: Check GPU Availability

In [4]:
import torch

print("="*80)
print("GPU AVAILABILITY")
print("="*80)

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"✅ CUDA GPU Detected")
    print(f"   Device: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    print(f"\n   Ray workers will share GPU using fractional allocation")
    print(f"   Example: 8 workers → each gets 0.125 GPU (1/8th of resources)")
    
elif torch.backends.mps.is_available():
    device = 'mps'
    print(f"✅ Apple MPS (Metal Performance Shaders) Detected")
    print(f"   Good performance for M1/M2/M3 Macs")
    
else:
    device = 'cpu'
    print(f"⚠️  No GPU detected - using CPU")
    print(f"\n   To enable GPU in Colab:")
    print(f"   1. Runtime → Change runtime type")
    print(f"   2. Hardware accelerator: GPU (T4 recommended)")
    print(f"   3. Save → Restart session")
    print(f"\n   Note: Training will be 50-100x slower on CPU")

print(f"\n   Default device: {device}")
print("="*80)

GPU AVAILABILITY
✅ CUDA GPU Detected
   Device: NVIDIA L4
   Memory: 23.7 GB

   Ray workers will share GPU using fractional allocation
   Example: 8 workers → each gets 0.125 GPU (1/8th of resources)

   Default device: cuda


## Step 5: Install Dependencies

In [5]:
# Install Python dependencies using uv
print("📦 Installing dependencies with uv...")
print("   This may take 1-2 minutes...\n")

!uv sync

print("\n✅ Dependencies installed successfully")

📦 Installing dependencies with uv...
   This may take 1-2 minutes...

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 130 packages in 0.99ms
Prepared 122 packages in 1m 00s                                          
Installed 122 packages in 264ms                             
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiohttp-cors==0.8.1
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + asttokens==3.0.1
 + attrs==25.4.0
 + certifi==2026.1.4
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + colorful==0.5.8
 + comm==0.2.3
 + contourpy==1.3.3
 + cryptography==46.0.5
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.3.4
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.2.1
 + distlib==0.4.0
 + einops==0.8.2
 + executing==2.2.1
 + filelock==3.24.3
 + fonttools==4.61.1
 + frozenlist==1.8.0
 + fsspec==2026.2.0
 + google-api-core==2.30.0
 + google-auth==2.48.0
 + googleapis-common-protos==1.72.0
 + grpcio==1.78.1
 + idna==

## Step 6: Verify Installation

In [6]:
# Verify key components
import sys
sys.path.insert(0, '/content/tako-v2')

print("="*80)
print("VERIFICATION")
print("="*80)

# Check imports
try:
    import torch
    print(f"✅ PyTorch: {torch.__version__}")
except ImportError as e:
    print(f"❌ PyTorch import failed: {e}")

try:
    import yaml
    print(f"✅ PyYAML installed")
except ImportError as e:
    print(f"❌ PyYAML import failed: {e}")

try:
    import ray
    print(f"✅ Ray: {ray.__version__}")
except ImportError as e:
    %pip install ray
    print(f"❌ Ray import failed: {e}")

try:
    from model.hrm import HRM
    print(f"✅ HRM model imported")
except ImportError as e:
    print(f"❌ HRM import failed: {e}")

try:
    from games.tictactoe import TicTacToeGame
    print(f"✅ TicTacToe game imported")
except ImportError as e:
    print(f"❌ TicTacToe import failed: {e}")

try:
    from training.mcts import MCTS
    print(f"✅ MCTS imported")
except ImportError as e:
    print(f"❌ MCTS import failed: {e}")

# Check directory structure
import os
required_dirs = ['model', 'games', 'training', 'config', 'scripts', 'checkpoints']
for d in required_dirs:
    if os.path.exists(d):
        print(f"✅ Directory: {d}/")
    else:
        print(f"❌ Directory missing: {d}/")

# Check config files
config_files = ['config/tictactoe.yaml', 'config/othello.yaml']
for cf in config_files:
    if os.path.exists(cf):
        print(f"✅ Config: {cf}")
    else:
        print(f"❌ Config missing: {cf}")

print("="*80)

VERIFICATION
✅ PyTorch: 2.10.0+cu128
✅ PyYAML installed
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 6.6 MB/s eta 0:00:000:00:0100:01
❌ Ray import failed: No module named 'ray'
✅ FlashAttention (SDPA) enabled
✅ HRM model imported
✅ TicTacToe game imported
✅ MCTS imported
✅ Directory: model/
✅ Directory: games/
✅ Directory: training/
✅ Directory: config/
✅ Directory: scripts/
✅ Directory: checkpoints/
✅ Config: config/tictactoe.yaml
✅ Config: config/othello.yaml


## Setup Complete! ✅

You're ready to use the Tako HRM notebooks.

### Next Steps:

1. **Train a model:** Open `train.ipynb` and run the training cell for your game
2. **Evaluate performance:** Open `eval.ipynb` to test win rates
3. **Play interactively:** Open `play.ipynb` to play against your model
4. **Benchmark MCTS:** Open `benchmark.ipynb` to measure performance

### Quick Training Example:

```python
# In train.ipynb:
# 1. Run TicTacToe training cell
# 2. Wait ~30 minutes
# 3. See 90%+ win rate vs random!
```

### Important Notes:

- **Checkpoints persist** in Google Drive (`/content/drive/MyDrive/tako_checkpoints/`)
- **Session state persists** - you only need to run this setup once per session
- **GPU recommended** - Training is 50-100x faster with GPU
- **Free tier works** - Colab free (T4 GPU) is sufficient for TicTacToe and Othello

---

**Happy training!** 🎮🤖